In [1]:
from pathlib import Path
from collections import defaultdict

import polars as pl

In [2]:
PLAYER_DIR = Path("data/player_hands")
ACTION_DIR = Path("data/actions")

MIN_HANDS = 500
MIN_3BET_OPPORTUNITIES = 100

In [3]:
stats = defaultdict(lambda: {
    "hands": 0,
    "three_bet_opportunities": 0,
    "three_bets": 0,
})

In [6]:

action_files = sorted(
    ACTION_DIR.glob("actions_*.parquet")
)

print(f"Found {len(action_files):,} action files")

Found 218 action files


In [7]:
for file_index, action_file in enumerate(action_files, start=1):

    chunk_id = action_file.stem.replace("actions_", "")

    player_file = (
        PLAYER_DIR /
        f"player_hands_{chunk_id}.parquet"
    )

    if not player_file.exists():
        print(f"Missing player file for {action_file.name}")
        continue

    print(
        f"[{file_index}/{len(action_files)}] "
        f"Processing chunk {chunk_id}"
    )

    # Load only needed columns
    players = pl.read_parquet(
        player_file,
        columns=[
            "hand_id",
            "player_id",
            "player_number",
        ],
    )

    actions = pl.read_parquet(
        action_file,
        columns=[
            "hand_id",
            "action_number",
            "raw_action",
        ],
    )

    # Count hands per player
    hand_counts = (
        players
        .group_by("player_id")
        .agg(
            pl.len().alias("hands")
        )
    )

    for player_id, hands in hand_counts.iter_rows():

        if player_id is not None:
            stats[player_id]["hands"] += hands

    # Map (hand_id, player_number) -> player_id
    player_lookup = {}

    for row in players.iter_rows(named=True):

        player_lookup[
            (
                row["hand_id"],
                row["player_number"],
            )
        ] = row["player_id"]

    # Group actions by hand
    hand_actions = (
        actions
        .sort([
            "hand_id",
            "action_number",
        ])
        .group_by(
            "hand_id",
            maintain_order=True,
        )
        .agg(
            pl.col("raw_action")
        )
    )

    # Identify 3-bet opportunities and 3-bets
    for hand_id, action_list in hand_actions.iter_rows():

        raises = 0

        for raw_action in action_list:

            parts = raw_action.split()

            if len(parts) < 2:
                continue

            # Flop dealt, so preflop is finished
            if parts[0] == "d" and parts[1] == "db":
                break

            # Ignore non-player actions
            if not parts[0].startswith("p"):
                continue

            try:
                player_number = int(parts[0][1:])
            except ValueError:
                continue

            action_type = parts[1]

            # First raise = open raise
            if action_type == "cbr" and raises == 0:
                raises = 1
                continue

            # After open raise, fold/call/raise all count
            # as a 3-bet opportunity
            if raises == 1 and action_type in {
                "f",
                "cc",
                "cbr",
            }:

                player_id = player_lookup.get(
                    (
                        hand_id,
                        player_number,
                    )
                )

                if player_id is None:
                    continue

                stats[player_id][
                    "three_bet_opportunities"
                ] += 1

                # Second raise = 3-bet
                if action_type == "cbr":

                    stats[player_id][
                        "three_bets"
                    ] += 1

                    # Anything after this would be
                    # facing a 3-bet, not an open raise
                    break

[1/218] Processing chunk 0000
[2/218] Processing chunk 0001
[3/218] Processing chunk 0002
[4/218] Processing chunk 0003
[5/218] Processing chunk 0004
[6/218] Processing chunk 0005
[7/218] Processing chunk 0006
[8/218] Processing chunk 0007
[9/218] Processing chunk 0008
[10/218] Processing chunk 0009
[11/218] Processing chunk 0010
[12/218] Processing chunk 0011
[13/218] Processing chunk 0012
[14/218] Processing chunk 0013
[15/218] Processing chunk 0014
[16/218] Processing chunk 0015
[17/218] Processing chunk 0016
[18/218] Processing chunk 0017
[19/218] Processing chunk 0018
[20/218] Processing chunk 0019
[21/218] Processing chunk 0020
[22/218] Processing chunk 0021
[23/218] Processing chunk 0022
[24/218] Processing chunk 0023
[25/218] Processing chunk 0024
[26/218] Processing chunk 0025
[27/218] Processing chunk 0026
[28/218] Processing chunk 0027
[29/218] Processing chunk 0028
[30/218] Processing chunk 0029
[31/218] Processing chunk 0030
[32/218] Processing chunk 0031
[33/218] Processi

In [8]:
rows = []

for player_id, values in stats.items():

    opportunities = values[
        "three_bet_opportunities"
    ]

    three_bets = values[
        "three_bets"
    ]

    three_bet_pct = (
        three_bets / opportunities * 100
        if opportunities > 0
        else None
    )

    rows.append({
        "player_id": player_id,
        "hands": values["hands"],
        "three_bet_opportunities": opportunities,
        "three_bets": three_bets,
        "three_bet_pct": three_bet_pct,
    })

player_stats = pl.DataFrame(rows)

player_stats.head()

player_id,hands,three_bet_opportunities,three_bets,three_bet_pct
str,i64,i64,i64,f64
"""6hXAotOLYqvlXW0qhMu1HA""",138,45,2,4.444444
"""2ePSN+HLDWDaDyFe0b2fYg""",464,129,1,0.775194
"""wjAHhXRJQJvcZCPXAvJGfg""",663,225,15,6.666667
"""Mhiwb9IWT0lQh96p/n5VHw""",7,3,0,0.0
"""bhsTNSwYevCwt8Re1HktfA""",118,47,3,6.382979


In [9]:
eligible = (
    player_stats
    .filter(
        (pl.col("hands") >= MIN_HANDS)
        &
        (
            pl.col("three_bet_opportunities")
            >= MIN_3BET_OPPORTUNITIES
        )
    )
    .sort(
        "three_bet_pct",
        descending=True,
    )
)

print("Total players:", player_stats.height)
print("Eligible players:", eligible.height)

eligible.head(20)

Total players: 278149
Eligible players: 34252


player_id,hands,three_bet_opportunities,three_bets,three_bet_pct
str,i64,i64,i64,f64
"""Rg+i50SS6+QeSQgE2FImPA""",1557,562,273,48.576512
"""mhifGhGiCbuMZM2R1vktCw""",1835,557,263,47.217235
"""AKyZn/1VZB+f9UIwtf2x2w""",563,223,101,45.29148
"""nJvYkAx04n101FyZf2GerQ""",511,161,70,43.478261
"""Zw3PjbL5JbMOXl36usuX/w""",574,151,63,41.721854
…,…,…,…,…
"""Lj1vXbOwCHCXVxzOfXgyJw""",516,196,70,35.714286
"""e/USI/aiy+oD1YNAsbogaQ""",660,171,61,35.672515
"""X4cAThyaVSJoJGgHhx4wKQ""",892,310,110,35.483871


In [10]:
summary = eligible.select([

    pl.len().alias(
        "eligible_players"
    ),

    pl.col("three_bet_pct")
    .mean()
    .alias("mean_player_3bet_pct"),

    pl.col("three_bet_pct")
    .median()
    .alias("median_player_3bet_pct"),

    pl.col("three_bet_pct")
    .std()
    .alias("std_player_3bet_pct"),

    pl.col("three_bet_pct")
    .quantile(0.25)
    .alias("q25_3bet_pct"),

    pl.col("three_bet_pct")
    .quantile(0.75)
    .alias("q75_3bet_pct"),
])

summary

eligible_players,mean_player_3bet_pct,median_player_3bet_pct,std_player_3bet_pct,q25_3bet_pct,q75_3bet_pct
u32,f64,f64,f64,f64,f64
34252,5.006977,4.091183,4.003817,2.352941,6.484432


In [11]:
pooled = eligible.select(

    (
        pl.col("three_bets").sum()
        /
        pl.col(
            "three_bet_opportunities"
        ).sum()
        * 100
    )
    .alias("pooled_3bet_pct")

)

pooled

pooled_3bet_pct
f64
5.508127


In [12]:
thresholds = [100, 250, 500, 1000]

for threshold in thresholds:

    filtered = player_stats.filter(
        (pl.col("hands") >= 500)
        &
        (pl.col("three_bet_opportunities") >= threshold)
    )

    result = filtered.select([
        pl.len().alias("players"),

        pl.col("three_bet_pct")
        .mean()
        .alias("mean_3bet"),

        pl.col("three_bet_pct")
        .median()
        .alias("median_3bet"),

        pl.col("three_bet_pct")
        .quantile(0.25)
        .alias("q25"),

        pl.col("three_bet_pct")
        .quantile(0.75)
        .alias("q75"),
    ])

    print(f"\nMinimum opportunities: {threshold}")
    print(result)


Minimum opportunities: 100
shape: (1, 5)
┌─────────┬───────────┬─────────────┬──────────┬──────────┐
│ players ┆ mean_3bet ┆ median_3bet ┆ q25      ┆ q75      │
│ ---     ┆ ---       ┆ ---         ┆ ---      ┆ ---      │
│ u32     ┆ f64       ┆ f64         ┆ f64      ┆ f64      │
╞═════════╪═══════════╪═════════════╪══════════╪══════════╡
│ 34252   ┆ 5.006977  ┆ 4.091183    ┆ 2.352941 ┆ 6.484432 │
└─────────┴───────────┴─────────────┴──────────┴──────────┘

Minimum opportunities: 250
shape: (1, 5)
┌─────────┬───────────┬─────────────┬──────────┬──────────┐
│ players ┆ mean_3bet ┆ median_3bet ┆ q25      ┆ q75      │
│ ---     ┆ ---       ┆ ---         ┆ ---      ┆ ---      │
│ u32     ┆ f64       ┆ f64         ┆ f64      ┆ f64      │
╞═════════╪═══════════╪═════════════╪══════════╪══════════╡
│ 24847   ┆ 5.092938  ┆ 4.22833     ┆ 2.513966 ┆ 6.594154 │
└─────────┴───────────┴─────────────┴──────────┴──────────┘

Minimum opportunities: 500
shape: (1, 5)
┌─────────┬───────────┬───────────